In [5]:
import pandas as pd

# Load summarized biomechanical profile
# summarized_bio = pd.read_parquet('../../data/processed/ml_datasets/summarized/final_summarized_biomechanical_profile_emg_added.parquet')
# print("\nSummarized Biomechanical Profile Columns and Unique Values:") 
# for col in summarized_bio.columns:
#     print(f"\n{col}:") 
#     print(f"Unique values: {summarized_bio[col].unique()}")

0

#  ranular Datasets:
# Resample/interpolate the biomechanics dataset to join evenly onto the EMG data: No changes
# Bio dataset with EMG filtered out dataset: is_interpolated filter: filter for non interpolated data if you want to take away emg data for a bio dataset without interpolated added columns (will filter out EMG so they are on the bio frequency)
# EMG dataset with phases added on: create a interpolated column list so we can differ that from the non and create a EMG dataset with pitch phases added on (creating the simplistic EMG dataset with phases added on, no fake data involved and straight to the muscles dataset)

# Load granular dataset with interpolated bio data
# granular_data = pd.read_parquet('../../data/processed/ml_datasets/final_inner_join_emg_biomech_data.parquet')
# print("\nGranular Dataset Columns and Unique Values:")
# for col in granular_data.columns:
#     print(f"\n{col}:")
#     print(f"Unique values: {granular_data[col].unique()}")

# Create pure EMG dataset without interpolated columns from biomech datasets


#load Non emg dataset so we can have all the trials for the variance analysis
# Load granular dataset with interpolated bio data
multi_trials_data = pd.read_parquet('../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet')
print("\nGranular Dataset Columns and Unique Values:")
for col in multi_trials_data.columns:
    print(f"\n{col}:")
    print(f"Unique values: {multi_trials_data[col].unique()}")





Granular Dataset Columns and Unique Values:

athlete_name:
Unique values: ['Everett Roberts' 'James Zajac' 'Jared Skolnicki' 'Brody Vigansky'
 'Josiah Miller' 'Luke Miller' 'Will Montgomerie' 'Josh Hejka'
 'Brady Pearson' 'Alex Edgar' 'Cameron Madsen' 'Leon Santiago']

athlete_dob:
Unique values: [datetime.date(1940, 1, 1) datetime.date(1994, 12, 20)
 datetime.date(1994, 7, 22) datetime.date(2010, 8, 11)
 datetime.date(2008, 3, 16) datetime.date(2009, 1, 8)
 datetime.date(1995, 6, 2) datetime.date(1997, 3, 20)
 datetime.date(2008, 2, 25) datetime.date(2008, 10, 28)
 datetime.date(2009, 5, 16) datetime.date(2008, 8, 24)]

athlete_traq:
Unique values: ['117060' '117670' '008904' '117032' '109567' '109568' '003616' '026522'
 '103495' '112587' '064450' '102879']

height_meters:
Unique values: [Decimal('1.8300') Decimal('1.9300') Decimal('1.8800') Decimal('1.8500')
 Decimal('1.8000') Decimal('1.7300') Decimal('1.7500')]

mass_kilograms:
Unique values: [Decimal('77.1100') Decimal('83.9200')

In [18]:
import pandas as pd
import numpy as np

# ----------------------
# 1) Load & define metrics
# ----------------------
multi_trials_data = pd.read_parquet('../../data/processed/ml_datasets/granular/granular_joint_details_2025-02-14.parquet')

kinematics_metrics = [
    'pitch_speed_mph',
    'shoulder_angle_x', 'shoulder_angle_y', 'shoulder_angle_z',
    'elbow_angle_x', 'elbow_angle_y', 'elbow_angle_z',
    'torso_angle_x', 'torso_angle_y', 'torso_angle_z',
    'pelvis_angle_x', 'pelvis_angle_y', 'pelvis_angle_z',
    'trunk_pelvis_dissociation'
]

kinetics_metrics = [
    'shoulder_velo_x', 'shoulder_velo_y', 'shoulder_velo_z',
    'elbow_velo_x', 'elbow_velo_y', 'elbow_velo_z',
    'torso_velo_x', 'torso_velo_y', 'torso_velo_z',
    'elbow_moment_x', 'elbow_moment_y', 'elbow_moment_z',
    'shoulder_thorax_moment_x', 'shoulder_thorax_moment_y', 'shoulder_thorax_moment_z',
    'max_shoulder_internal_rotational_velo'
]

energetics_metrics = [
    'shoulder_energy_transfer', 'shoulder_energy_generation',
    'elbow_energy_transfer', 'elbow_energy_generation',
    'lead_knee_energy_transfer', 'lead_knee_energy_generation'
]

all_metrics = kinematics_metrics + kinetics_metrics + energetics_metrics

# ----------------------
# 2) Define settings
# ----------------------
athletes = multi_trials_data['athlete_name'].unique()
target_date = pd.to_datetime("2025-02-14").date()

n_subsets = 3
subset_size = 3

# ----------------------
# 3) Helper functions
# ----------------------
def sample_metrics(trial_ids, summary_df, metrics, n=subset_size):
    selected_trials = np.random.choice(trial_ids, size=n, replace=False)
    subset = summary_df[summary_df['session_trial'].isin(selected_trials)]
    return subset[metrics].mean()

def compute_outlier_stats(series):
    if len(series) < 2:
        return series.var(), 0  # Not enough data
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    non_outliers = series[(series >= lower_bound) & (series <= upper_bound)]
    n_outliers = len(series) - len(non_outliers)
    variance_cleaned = non_outliers.var() if len(non_outliers) > 1 else np.nan
    return variance_cleaned, n_outliers

# ----------------------
# 4) Generate variance_summary_df
#    (Random subsets + outlier stats)
# ----------------------
results = []
for athlete in athletes:
    athlete_data = multi_trials_data[
        (multi_trials_data['athlete_name'] == athlete) &
        (multi_trials_data['session_date'] == target_date)
    ]
    
    phases = athlete_data['pitch_phase'].dropna().unique()
    if len(phases) == 0:
        print(f"No pitch_phase data for {athlete}. Skipping pitch phase grouping.")
        continue

    for phase in phases:
        phase_data = athlete_data[athlete_data['pitch_phase'] == phase]
        trial_summary = phase_data.groupby('session_trial')[all_metrics].mean().reset_index()
        
        unique_trials = trial_summary['session_trial'].unique()
        n_trials = len(unique_trials)
        
        if n_trials < 5:
            print(f"Skipping {athlete} for pitch phase '{phase}' due to insufficient trials (found {n_trials}).")
            continue
        
        domain_reports = {
            'kinematics': [],
            'kinetics': [],
            'energetics': []
        }
        
        # Build random subsets for each domain
        for i in range(n_subsets):
            rep_kin = sample_metrics(unique_trials, trial_summary, kinematics_metrics, n=subset_size)
            rep_kinet = sample_metrics(unique_trials, trial_summary, kinetics_metrics, n=subset_size)
            rep_energ = sample_metrics(unique_trials, trial_summary, energetics_metrics, n=subset_size)
            
            domain_reports['kinematics'].append(rep_kin)
            domain_reports['kinetics'].append(rep_kinet)
            domain_reports['energetics'].append(rep_energ)
        
        # Compute variance, outlier stats
        result_entry = {
            'athlete_name': athlete,
            'pitch_phase': phase,
            'n_trials': n_trials
        }
        
        for domain, reports in domain_reports.items():
            reports_df = pd.DataFrame(reports)
            domain_variance = reports_df.var()
            
            for metric in reports_df.columns:
                orig_var = domain_variance.get(metric, np.nan)
                cleaned_var, n_outliers = compute_outlier_stats(reports_df[metric])
                var_diff = orig_var - cleaned_var if pd.notnull(cleaned_var) else np.nan
                
                result_entry[f"{domain}_{metric}_variance"] = orig_var
                result_entry[f"{domain}_{metric}_variance_after_outlier_removal"] = cleaned_var
                result_entry[f"{domain}_{metric}_n_outliers"] = n_outliers
                result_entry[f"{domain}_{metric}_outlier_removal_diff"] = var_diff
        
        results.append(result_entry)

variance_summary_df = pd.DataFrame(results)

print("\nVariance summary (with outlier stats) across domains for each athlete and pitch phase:")
print(variance_summary_df.head())
print("\nColumns:", variance_summary_df.columns.tolist())

# ----------------------
# 5) Overall summary from granular data
#    to avoid 'averaging the averages'
# ----------------------
granular_filtered = multi_trials_data[
    (multi_trials_data['session_date'] == target_date) &
    (multi_trials_data['pitch_phase'].notna())  # optional: if you only want pitched phases
].copy()

# Convert all the metrics columns to float so they are recognized as numeric
for col in all_metrics:
    if col in granular_filtered.columns:
        granular_filtered.loc[:, col] = granular_filtered[col].astype(float)

# Now, numeric_metrics can be defined directly as all_metrics.
numeric_metrics = all_metrics
print(f"Numeric columns they say = ================= {numeric_metrics}")

overall_phase_report = (
    granular_filtered
    .groupby('athlete_name')[numeric_metrics]  # group by athlete_name
    .mean()                                    # compute mean directly from the granular data
    .reset_index()
)

print("\nOverall Metric Averages (from granular data) grouped by athlete_name:")
print(overall_phase_report)



Skipping Jared Skolnicki for pitch phase 'Wind-Up' due to insufficient trials (found 1).
Skipping Jared Skolnicki for pitch phase 'Stride' due to insufficient trials (found 1).
Skipping Jared Skolnicki for pitch phase 'Arm Cocking' due to insufficient trials (found 1).
Skipping Jared Skolnicki for pitch phase 'Arm Acceleration' due to insufficient trials (found 1).
Skipping Jared Skolnicki for pitch phase 'Follow Through' due to insufficient trials (found 1).

Variance summary (with outlier stats) across domains for each athlete and pitch phase:
      athlete_name       pitch_phase  n_trials  \
0  Everett Roberts           Wind-Up         7   
1  Everett Roberts            Stride         7   
2  Everett Roberts       Arm Cocking         7   
3  Everett Roberts  Arm Acceleration         7   
4  Everett Roberts    Follow Through         7   

   kinematics_pitch_speed_mph_variance  \
0                             0.104815   
1                             0.013333   
2                    